<a href="https://colab.research.google.com/github/Solo7602/web/blob/2lab/laba2_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.corpus import stopwords
from pymorphy3 import MorphAnalyzer
import re

# --- стоп-слова ---
nltk.download("stopwords")
stopwords_ru = stopwords.words("russian")

# --- морфология ---
morph = MorphAnalyzer()

# --- очистка html ---
HTML_TAG_RE = re.compile(r"<[^>]+>")

def clean_html(text):
    if not isinstance(text, str):
        return ""
    return HTML_TAG_RE.sub("", text).strip()


# --- лемматизация текста ---
def lemmatize_text(text):
    if not isinstance(text, str):
        return ""

    text = clean_html(text)
    tokens = []
    for w in re.findall(r"[А-Яа-яA-Za-z]+", text):
        w = w.lower()
        if w in stopwords_ru:
            continue
        lemma = morph.normal_forms(w)[0]
        tokens.append(lemma)

    return " ".join(tokens)


# ============================
#        MAIN CODE
# ============================

df = pd.read_csv("sample_data/hh_vacancies_details.csv")

# создаём новый столбец с нормализованным текстом
df["text_processed"] = df["description_clean"].apply(lemmatize_text)

# векторизация
cv = CountVectorizer(max_features=5000)
X = cv.fit_transform(df["text_processed"])

# LDA
lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method="batch"
)
lda.fit(X)

words = cv.get_feature_names_out()

# вывод тем
print("\nТемы LDA:")
for i, topic in enumerate(lda.components_):
    top_words = [words[j] for j in topic.argsort()[-15:]]  # 15 слов
    print(f"\nТема {i+1}:")
    print(", ".join(top_words))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Темы LDA:

Тема 1:
работать, анализ, сотрудник, уровень, требование, процесс, знание, наш, бизнес, команда, условие, данные, опыт, компания, работа

Тема 2:
возможность, система, решение, заказчик, так, бизнес, задача, проектный, требование, проект, опыт, процесс, компания, erp, работа

Тема 3:
корпоративный, обучение, знание, требование, сотрудник, проект, возможность, система, процесс, quot, анализ, опыт, компания, бизнес, работа

Тема 4:
система, команда, учёт, знание, решение, проект, задача, год, контроль, финансовый, анализ, опыт, компания, quot, работа

Тема 5:
работать, аналитический, компания, умение, процесс, команда, требование, разработка, знание, проект, данные, бизнес, опыт, анализ, работа


In [4]:
pip install pymorphy3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 24.5 MB/s eta 0:00:00
